In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import requests
import numpy as np
import pandas as pd

from ollama import chat, embed
from ollama import ChatResponse


In [ ]:
prompt = "What is the candidate's name?"

In [ ]:
documents = []

## Docs for RAG

In [ ]:
# List of URLs to load documents from
urls = [
    "https://www.linkedin.com/in/ciaran-mcdonnell95/",
    
]
# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

In [ ]:
with open("data/Plain_cv.txt", encoding="utf-8") as cv:
    context = cv.readlines()

documents.append(" ".join(context).replace("\n", " "))
    

## Next embed the documents

In [ ]:
def get_embedding(text, model="llama3.2:1b"):
    return embed(model=model, input=text).embeddings[0]

In [ ]:
model = "llama3.2:1b"

In [ ]:
df_embeddings = pd.DataFrame(data=documents,columns=["text"]).assign(
    embeddings = lambda df: df["text"].apply(get_embedding),
    dot = lambda df: np.dot(list(df["embeddings"].values), get_embedding(prompt))
    )
df_embeddings


## Now the full prompting

In [ ]:
system_setup = """You are an assistant that is helping a recruiter to answer questions about a job candidate based on their CV. Always answer in the third person.
Please make the candidate seem as impressive as possible. Answer in as few words as possible."""

assistant_directives = """
Use this information from the candidate's CV to answer the user question: {context}. All information in this document may be shared. Do not under any circumstances lie about the candidate.
If you think the question has been answered stop.
"""

In [ ]:
df_embeddings

In [ ]:
def question_cv(question:str, documents:list[str], model:str="llama3.2:1b", ) -> None:
    df_embeddings = (
        pd.DataFrame(
            data=documents, columns=["text"]
        )
        .assign(
            embeddings = lambda df: df["text"].apply(get_embedding),
            dot = lambda df: np.dot(list(df["embeddings"].values), get_embedding(question))
            )
        .sort_values(
            by="dot",
            ascending=False
            )
    )

    response = chat(model=model, 
    messages=[
        {"role": "system", "content": system_setup},
        { 'role': 'assistant', 'content': assistant_directives.format(context=df_embeddings["text"].iloc[0])},
        { 'role': 'user', 'content': question},
    ])

    print(response.message.content)
        


In [ ]:
question= "Does the candidate have any experience in finance?"

In [ ]:
question_cv(question=question, documents=documents, model="llama3.2:1b")

In [1]:
from cv_bot.llm_assistant import CVAssistant

In [2]:
documents = []
with open("data/Plain_cv.txt", encoding="utf-8") as cv:
    context = cv.readlines()

documents.append(" ".join(context).replace("\n", " "))

cv_assist = CVAssistant(model="llama3.2:1b", documents=documents)

In [4]:
question= "Does the candidate have any experience in finance?"

cv_assist.print_response(question=question)

Based on the provided CV, it appears that the candidate has limited experience in finance. There are no mentions of financial analysis, portfolio management, or experience working with financial data.

However, it is worth noting that the candidate has experience in data science and machine learning, which are fields closely related to finance. They have worked on projects such as:

* Developing automated ETL pipelines to extract, transform, and load data from multiple sources
* Creating algorithms for task automation and machine learning
* Presenting findings to senior management and recommending strategic next steps for the company's data science department

Additionally, the candidate has experience in financial analysis tools such as Excel, Tableau, and Python libraries like Pandas and NumPy.

To address the question of whether the candidate has experience in finance, we can consider their skills and expertise in the following areas:

* Financial analysis: The candidate has demonst